# 🎓 Visoria: Attention Detection & Explainable AI (XAI) Model
### An Attention-Aware Socratic Pedagogical Intervention System
**Researchers:** Omor Faruk (2202026), Mostafa Shariare (2202045)  
**Supervisor:** Farhana Islam, Assistant Professor  
**Affiliation:** Department of Educational Technology and Engineering, University of Frontier Technology, Bangladesh

---

### 📌 Overview & Colab Execution Guide
This notebook implements the complete research-grade Machine Learning & Explainable AI (XAI) pipeline for **Visoria**:
1. **Multi-Feature Fusion**: Estimates attention from computer vision telemetry (facial coordinates & confidence, head/body pose, hand count, and mobile phone detection).
2. **5-Fold Stratified Cross-Validation**: Compares Logistic Regression, Support Vector Classifiers (SVC), Random Forest, and XGBoost.
3. **Hyperparameter Optimization**: Tunes the top-performing ensemble for optimal F1-score and generalization.
4. **Probability Calibration**: Uses Platt Scaling (`CalibratedClassifierCV`) so confidence scores match real-world empirical risk (essential for the live student telemetry dashboard).
5. **Ablation Study**: Rigorously evaluates model robustness when dominant signals (pose/phone) are excluded.
6. **Explainable AI (XAI)**: Provides global and local interpretations via **SHAP** (TreeExplainer) and **LIME** (Local Interpretable Model-agnostic Explanations).
7. **Artifact Export & 1-Click Download**: Bundles `attention_model.pkl`, `attention_scaler.pkl`, and `attention_columns.pkl` into a downloadable zip file ready to drop into the Visoria backend.

💡 **Quick Start in Google Colab:** Click **Runtime** → **Run all** (`Ctrl + F9`).


In [ ]:
# 1. Install required packages in Google Colab
!pip install -q xgboost imbalanced-learn shap lime joblib scikit-learn matplotlib seaborn


In [ ]:
# 2. Imports and publication-grade plot styling
import os, sys, warnings, json, io, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, brier_score_loss
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import shap
from lime.lime_tabular import LimeTabularExplainer
import joblib

# Set aesthetic publication defaults
plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 10.5,
    "axes.titlesize": 12.5,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ARTIFACTS_DIR = "artifacts"
OUT_DIR = "outputs"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("✓ Environment configured and libraries loaded successfully.")


## 📂 2. Data Ingestion
Upload `attention_detection_dataset_v1.csv` directly into Colab, or the cell will auto-detect the file if it has already been uploaded.


In [ ]:
# 3. Load dataset (Auto-detect, Colab file upload prompt, or synthetic fallback)
DATASET_FILENAME = "attention_detection_dataset_v1.csv"

# Check standard and nested candidate paths
candidate_paths = [
    DATASET_FILENAME,
    f"data/{DATASET_FILENAME}",
    f"ml/data/{DATASET_FILENAME}",
    f"../data/{DATASET_FILENAME}",
    f"../ml/data/{DATASET_FILENAME}"
]

found_path = None
for p in candidate_paths:
    if os.path.exists(p):
        found_path = p
        break

if not found_path:
    try:
        from google.colab import files
        print(f"Please select and upload '{DATASET_FILENAME}' from your computer:")
        uploaded = files.upload()
        if DATASET_FILENAME in uploaded:
            found_path = DATASET_FILENAME
    except Exception as e:
        print("Note: Running outside interactive Colab upload or upload bypassed.")

if not found_path or not os.path.exists(found_path):
    print("⚠️ Dataset not found on disk. Generating an empirical demonstration dataset based on Visoria telemetry distributions...")
    n_samples = 2000
    poses = np.random.choice(["forward", "down", "left", "right"], size=n_samples, p=[0.50, 0.25, 0.15, 0.10])
    phones = np.random.choice([0, 1], size=n_samples, p=[0.82, 0.18])
    faces = np.random.choice([1, 2, 0], size=n_samples, p=[0.92, 0.06, 0.02])
    hands = np.random.choice([0, 1, 2], size=n_samples, p=[0.75, 0.18, 0.07])
    
    face_w = np.random.normal(160, 25, size=n_samples).clip(80, 260)
    face_h = face_w * np.random.normal(1.0, 0.05, size=n_samples)
    face_x = np.random.normal(280, 40, size=n_samples)
    face_y = np.random.normal(200, 30, size=n_samples)
    face_con = np.random.normal(86, 8, size=n_samples).clip(50, 98)
    
    pose_x = np.random.normal(0, 15, size=n_samples)
    pose_y = np.random.normal(0, 12, size=n_samples)
    
    phone_x = np.where(phones == 1, np.random.uniform(100, 300, n_samples), 0)
    phone_y = np.where(phones == 1, np.random.uniform(100, 400, n_samples), 0)
    phone_w = np.where(phones == 1, np.random.uniform(80, 200, n_samples), 0)
    phone_h = np.where(phones == 1, np.random.uniform(120, 300, n_samples), 0)
    phone_con = np.where(phones == 1, np.random.uniform(0.6, 0.95, n_samples), 0)
    
    # Ground truth label: 1 = Attentive, 0 = Distracted
    # Strong correlation with forward pose, no phone, stable face
    logit = (
        2.2 * (poses == "forward")
        - 3.5 * phones
        - 1.8 * (poses == "down")
        - 1.2 * (poses == "left")
        - 1.2 * (poses == "right")
        + 0.03 * (face_con - 80)
        - 0.5 * (hands > 1)
    )
    prob_attentive = 1 / (1 + np.exp(-logit))
    labels = (np.random.rand(n_samples) < prob_attentive).astype(int)
    
    df = pd.DataFrame({
        "no_of_face": faces, "face_x": face_x, "face_y": face_y,
        "face_w": face_w, "face_h": face_h, "face_con": face_con,
        "no_of_hand": hands, "pose": poses, "pose_x": pose_x, "pose_y": pose_y,
        "phone": phones, "phone_x": phone_x, "phone_y": phone_y,
        "phone_w": phone_w, "phone_h": phone_h, "phone_con": phone_con,
        "label": labels
    })
    df.to_csv(DATASET_FILENAME, index=False)
    print(f"✓ Created working dataset: {DATASET_FILENAME}")
else:
    df = pd.read_csv(found_path)
    print(f"✓ Loaded dataset from: {found_path}")

print(f"  Total samples: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()


## 📊 3. Exploratory Data Analysis (EDA)
Examine class balance, missing values, and behavioral telemetry distributions.


In [ ]:
# 4. Summary statistics & distribution visualizations
print("Missing values per column:\n", df.isna().sum())
print("\nTarget Label Distribution:")
print(df['label'].value_counts())
print("\nClass Balance Proportions:")
print(df['label'].value_counts(normalize=True).rename('Proportion').round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Target class distribution
sns.countplot(x="label", data=df, ax=axes[0], palette=["#EF4444", "#22C55E"])
axes[0].set_title("Class Balance (0: Distracted, 1: Attentive)")
axes[0].set_xticklabels(["Distracted (0)", "Attentive (1)"])

# Head pose vs Attention
sns.countplot(x="pose", hue="label", data=df, ax=axes[1], palette=["#EF4444", "#22C55E"])
axes[1].set_title("Pose Orientation vs Attention State")
axes[1].tick_params(axis='x', rotation=20)

# Phone presence vs Attention
sns.countplot(x="phone", hue="label", data=df, ax=axes[2], palette=["#EF4444", "#22C55E"])
axes[2].set_title("Phone Presence vs Attention State")
axes[2].set_xticklabels(["No Phone (0)", "Phone Detected (1)"])

plt.tight_layout()
plt.show()


## 🔍 4. Heuristic Baseline Check
Analyze the empirical separability of dominant features (`phone` and `pose`) before model training.


In [ ]:
# 5. Evaluate trivial heuristic rule vs ground truth
phone_active = df.loc[df.phone == 1, "label"]
print("Distraction rate when phone is detected:")
print(phone_active.value_counts(normalize=True).round(4).to_dict())

rule_pred = ((df.pose == "forward") & (df.phone == 0)).astype(int)
rule_acc = accuracy_score(df.label, rule_pred)
rule_f1 = f1_score(df.label, rule_pred)
print(f"\nTrivial 2-feature rule baseline (Forward Pose AND No Phone):")
print(f"  Accuracy: {rule_acc:.4f} | F1-Score: {rule_f1:.4f}")


## ⚙️ 5. Preprocessing & Feature Engineering
- Derive geometric features: `face_area` ($w \times h$) and `face_aspect_ratio` ($w / h$).
- One-hot encode categorical `pose`.
- Split into Stratified Train/Test partitions (80% / 20%).
- Standardize numerical scale with `StandardScaler`.


In [ ]:
# 6. Feature transformation & alignment pipeline
def preprocess_features(df_in, fit_columns=None):
    d = df_in.copy().fillna(0)
    
    # Feature engineering: facial bounding dimensions
    d["face_area"] = d["face_w"] * d["face_h"]
    d["face_aspect_ratio"] = d["face_w"] / (d["face_h"] + 1e-6)
    
    # One-hot encode pose orientation
    if "pose" in d.columns:
        d = pd.get_dummies(d, columns=["pose"], drop_first=False)
        
    y_target = d.pop("label") if "label" in d.columns else None
    
    # Align columns to strict schema
    if fit_columns is not None:
        for c in fit_columns:
            if c not in d.columns:
                d[c] = 0
        d = d[fit_columns]
    
    return d, y_target

X_full, y = preprocess_features(df)
feature_names = list(X_full.columns)

print(f"✓ Engineered {len(feature_names)} features:")
for i, f_name in enumerate(feature_names):
    print(f"  {i+1:02d}. {f_name}")

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"\n✓ Training set: {X_train.shape[0]} samples")
print(f"✓ Testing set:  {X_test.shape[0]} samples")


## 🏆 6. Multi-Model Benchmark (5-Fold Stratified CV)
Compare 4 candidate architectures under identical cross-validation folds:
1. **Logistic Regression** (Linear baseline with balanced weights)
2. **Support Vector Machine (SVC)** (RBF non-linear kernel)
3. **Random Forest Classifier** (Bagged ensemble)
4. **XGBoost Classifier** (Gradient-boosted decision trees)


In [ ]:
# 7. 5-Fold Stratified Cross-Validation benchmark
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"),
    "SVC (RBF)": SVC(probability=True, random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0)
}

results = []
print("Evaluating candidate models with 5-fold Stratified CV...")
for name, m in candidate_models.items():
    scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    scores = cross_validate(m, X_train_s, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy": np.mean(scores["test_accuracy"]),
        "Precision": np.mean(scores["test_precision"]),
        "Recall": np.mean(scores["test_recall"]),
        "F1-Score": np.mean(scores["test_f1"]),
        "F1-Std": np.std(scores["test_f1"]),
        "ROC-AUC": np.mean(scores["test_roc_auc"]),
    })

bench_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
display(bench_df.style.highlight_max(axis=0, color="#dcfce7"))

# Benchmark Bar Chart
fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ["#22C55E", "#8B5CF6", "#3B82F6", "#F59E0B"]
bars = ax.bar(bench_df["Model"], bench_df["F1-Score"], yerr=bench_df["F1-Std"], capsize=5, color=colors[:len(bench_df)], width=0.55)
ax.set_ylim(0.70, 1.02)
ax.set_ylabel("5-Fold Mean F1-Score")
ax.set_title("Model Architecture Benchmark (5-Fold Stratified CV)")
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold', fontsize=9.5)
plt.tight_layout()
plt.show()


## 🎯 7. Hyperparameter Tuning
Optimize hyperparameters using `RandomizedSearchCV` to maximize F1 score.


In [ ]:
# 8. Randomized Search for optimal hyperparameters
best_model_name = bench_df.iloc[0]["Model"]
print(f"Optimizing hyperparameters for top candidate: {best_model_name}")

if "Random Forest" in best_model_name:
    estimator = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")
    param_dist = {
        "n_estimators": [100, 200, 300, 400],
        "max_depth": [None, 8, 12, 16, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5]
    }
elif "XGBoost" in best_model_name:
    estimator = XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0)
    param_dist = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7, 9],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0]
    }
else:
    estimator = candidate_models[best_model_name]
    param_dist = {}

if param_dist:
    search = RandomizedSearchCV(
        estimator,
        param_distributions=param_dist,
        n_iter=15,
        cv=cv,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train_s, y_train)
    print(f"✓ Best CV F1-Score: {search.best_score_:.4f}")
    print("✓ Best Parameters:", search.best_params_)
    tuned_model = search.best_estimator_
else:
    estimator.fit(X_train_s, y_train)
    tuned_model = estimator


## 🧪 8. Final Held-Out Test Set Evaluation
Assess final generalization metrics on the 20% untouched holdout split.


In [ ]:
# 9. Test set evaluation and diagnostic curves
y_pred = tuned_model.predict(X_test_s)
y_proba = tuned_model.predict_proba(X_test_s)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=" * 45)
print("       HELD-OUT TEST SET METRICS")
print("=" * 45)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Distracted", "Attentive"]))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Distracted", "Attentive"]).plot(
    ax=axes[0], cmap="Blues", values_format="d"
)
axes[0].set_title("Held-Out Confusion Matrix")
axes[0].grid(False)

# ROC Curve
RocCurveDisplay.from_estimator(tuned_model, X_test_s, y_test, ax=axes[1], color="#22C55E", name=best_model_name)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Chance")
axes[1].set_title(f"ROC Curve (AUC = {auc:.4f})")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## ⚖️ 9. Probability Calibration (Confidence Reliability)
The Visoria Student App and Teacher Dashboard display live attention confidence percentages.
Uncalibrated tree ensembles are often overconfident. We apply **Platt Scaling** via `CalibratedClassifierCV` to minimize Brier score loss.


In [ ]:
# 10. Probability Calibration & Reliability Diagram
cal_model = CalibratedClassifierCV(tuned_model, method="sigmoid", cv="prefit")
cal_model.fit(X_train_s, y_train)

y_proba_cal = cal_model.predict_proba(X_test_s)[:, 1]

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print("Brier Score Loss (lower is better):")
print(f"  Uncalibrated: {brier_raw:.4f}")
print(f"  Calibrated:   {brier_cal:.4f}")

prob_true_raw, prob_pred_raw = calibration_curve(y_test, y_proba, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.5)
plt.plot(prob_pred_raw, prob_true_raw, "s-", color="#EF4444", label=f"Uncalibrated (Brier={brier_raw:.3f})")
plt.plot(prob_pred_cal, prob_true_cal, "o-", color="#22C55E", label=f"Calibrated (Brier={brier_cal:.3f})")
plt.xlabel("Mean Predicted Confidence")
plt.ylabel("Observed Fraction of Positives")
plt.title("Reliability Calibration Curve")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Set calibrated model for production deployment
final_model = cal_model


## 🔬 10. Ablation Study: Generalization without `pose` & `phone`
Test whether secondary visual telemetry (face scale, aspect ratio, hand activity, confidence) maintains predictive power when dominant cues are omitted.


In [ ]:
# 11. Ablation experiment (masking pose and phone features)
abl_cols = [c for c in feature_names if not (c.startswith("pose_") or c == "phone" or c.startswith("phone_"))]
print(f"Ablated feature subset ({len(abl_cols)} features):", abl_cols)

X_train_abl = X_train[abl_cols]
X_test_abl = X_test[abl_cols]

scaler_abl = StandardScaler()
X_tr_abl_s = scaler_abl.fit_transform(X_train_abl)
X_te_abl_s = scaler_abl.transform(X_test_abl)

abl_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")
abl_model.fit(X_tr_abl_s, y_train)

y_pred_abl = abl_model.predict(X_te_abl_s)
abl_f1 = f1_score(y_test, y_pred_abl)
abl_acc = accuracy_score(y_test, y_pred_abl)

print(f"Full Model F1-Score:    {f1:.4f}")
print(f"Ablated Model F1-Score: {abl_f1:.4f} (without pose and phone)")

plt.figure(figsize=(6, 4))
plt.bar(["Full Model", "Ablated (No Pose/Phone)"], [f1, abl_f1], color=["#22C55E", "#F59E0B"], width=0.45)
plt.ylabel("Test F1-Score")
plt.ylim(0, 1.1)
plt.title("Ablation Study: Feature Generalization Drop")
for i, v in enumerate([f1, abl_f1]):
    plt.text(i, v + 0.03, f"{v:.4f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 🧠 11. Explainable AI (XAI) Suite: SHAP & LIME
To make automated pedagogical interventions trustworthy and transparent for instructors and students, we extract both **global** and **local** feature attributions.


In [ ]:
# 12. SHAP Global Explanations
# Use the underlying tree estimator for SHAP
tree_estimator = tuned_model if not hasattr(tuned_model, "estimator") else tuned_model.estimator

try:
    explainer = shap.TreeExplainer(tree_estimator)
    shap_sample = X_test_s[:200]
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list):
        sv = shap_values[1]  # Attentive class
    elif len(shap_values.shape) == 3:
        sv = shap_values[:, :, 1]
    else:
        sv = shap_values

    plt.figure(figsize=(9.5, 5.5))
    shap.summary_plot(sv, shap_sample, feature_names=feature_names, show=False)
    plt.title("SHAP Global Feature Importance (Attentive Class)", fontsize=13, pad=15)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("SHAP TreeExplainer note:", e)


In [ ]:
# 13. LIME Local Explanations (Single-Sample Attribution)
lime_explainer = LimeTabularExplainer(
    training_data=X_train_s,
    feature_names=feature_names,
    class_names=["Distracted", "Attentive"],
    mode="classification",
    random_state=RANDOM_STATE
)

# Identify one attentive sample and one distracted sample from test set
idx_att = np.where(y_test.values == 1)[0][0]
idx_dist = np.where(y_test.values == 0)[0][0]

exp_att = lime_explainer.explain_instance(X_test_s[idx_att], tuned_model.predict_proba, num_features=6)
exp_dist = lime_explainer.explain_instance(X_test_s[idx_dist], tuned_model.predict_proba, num_features=6)

print("LIME Explanation for Attentive Student Sample:")
for feat, weight in exp_att.as_list():
    direction = "Supports Focus" if weight > 0 else "Suggests Distraction"
    print(f"  {feat:<35}: {weight:+.4f} ({direction})")

print("\nLIME Explanation for Distracted Student Sample:")
for feat, weight in exp_dist.as_list():
    direction = "Supports Focus" if weight > 0 else "Suggests Distraction"
    print(f"  {feat:<35}: {weight:+.4f} ({direction})")


## 💾 12. Model Artifact Export & Download
Saves the three production artifacts required by Visoria (`attention_model.pkl`, `attention_scaler.pkl`, `attention_columns.pkl`) and packages them as a downloadable `.zip` file.


In [ ]:
# 14. Save artifacts and generate download link
joblib.dump(final_model, f"{ARTIFACTS_DIR}/attention_model.pkl")
joblib.dump(scaler, f"{ARTIFACTS_DIR}/attention_scaler.pkl")
joblib.dump(feature_names, f"{ARTIFACTS_DIR}/attention_columns.pkl")

summary_metrics = {
    "model_architecture": best_model_name,
    "test_accuracy": float(acc),
    "test_precision": float(prec),
    "test_recall": float(rec),
    "test_f1": float(f1),
    "test_roc_auc": float(auc),
    "brier_uncalibrated": float(brier_raw),
    "brier_calibrated": float(brier_cal),
    "ablation_f1_drop": float(f1 - abl_f1)
}

with open(f"{OUT_DIR}/visoria_model_summary.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

print("✓ Saved serialized artifacts to directory:", ARTIFACTS_DIR)
print(json.dumps(summary_metrics, indent=2))

# Archive artifacts for 1-click download
zip_path = "visoria_model_artifacts"
shutil.make_archive(zip_path, 'zip', ARTIFACTS_DIR)
print(f"\n✓ Artifact bundle created: {zip_path}.zip")

# Trigger browser download if running in Colab
try:
    from google.colab import files
    print("Initiating automatic download of model artifacts...")
    files.download(f"{zip_path}.zip")
except Exception:
    print(f"Download manually or access from local file path: {zip_path}.zip")


## 🚀 13. Real-Time Inference Interface
Simulates how the Visoria backend and Student App consume the trained model during active live sessions.


In [ ]:
# 15. Inference function demonstration
def predict_live_attention(raw_telemetry: dict):
    """Simulates production prediction from raw camera telemetry."""
    df_raw = pd.DataFrame([raw_telemetry]).fillna(0)
    
    # Feature engineering
    df_raw["face_area"] = df_raw.get("face_w", 0) * df_raw.get("face_h", 0)
    df_raw["face_aspect_ratio"] = df_raw.get("face_w", 0) / (df_raw.get("face_h", 1) + 1e-6)
    
    if "pose" in df_raw.columns:
        df_raw = pd.get_dummies(df_raw, columns=["pose"])
        
    df_aligned = pd.DataFrame(0, index=[0], columns=feature_names)
    for col in feature_names:
        if col in df_raw.columns:
            df_aligned[col] = df_raw[col].values
            
    X_scaled = scaler.transform(df_aligned.astype(float))
    prediction = int(final_model.predict(X_scaled)[0])
    confidence = float(final_model.predict_proba(X_scaled)[0, 1])
    
    return {
        "status": "Attentive" if prediction == 1 else "Distracted",
        "attention_score_pct": round(confidence * 100, 1),
        "pedagogical_intervention_needed": confidence < 0.60
    }

# Case A: Attentive learner facing screen
learner_focused = {
    "no_of_face": 1, "face_x": 250, "face_y": 140, "face_w": 165, "face_h": 165, "face_con": 91.2,
    "no_of_hand": 0, "pose": "forward", "pose_x": 0.2, "pose_y": 0.1,
    "phone": 0, "phone_x": 0, "phone_y": 0, "phone_w": 0, "phone_h": 0, "phone_con": 0
}

# Case B: Distracted learner looking away on phone
learner_distracted = {
    "no_of_face": 1, "face_x": 256, "face_y": 246, "face_w": 141, "face_h": 141, "face_con": 89.0,
    "no_of_hand": 2, "pose": "down", "pose_x": 32.5, "pose_y": 36.5,
    "phone": 1, "phone_x": 117, "phone_y": 137, "phone_w": 276, "phone_h": 421, "phone_con": 0.88
}

print("Inference Test Case 1 (Focused Student):")
print(json.dumps(predict_live_attention(learner_focused), indent=2))

print("\nInference Test Case 2 (Distracted Student with Phone):")
print(json.dumps(predict_live_attention(learner_distracted), indent=2))
